# §5 — VLM injection-strategy comparison on CLEVR

Trains the VLM (CLIP-pretrained ViT + projector + SmolLM2-360M-Instruct) for **500 steps** with three injection strategies (`cls`, `all_patches`, `interleaved`). ViT and decoder are frozen — only the projector trains. Reports val exact-match accuracy on up to 500 CLEVR-val examples, visual tokens per example, peak GPU memory, and seconds/step.

**Prereqs:**
1. `runs/clip_eurosat/best.pt` from §3 must already exist on Drive.
2. CLEVR-mini data lives at `data/clevr_mini/{train,val}.jsonl` + `data/clevr_mini/images/`. The notebook runs `scripts/download_clevr.py` if it's missing.

**Compute:** 3 runs × 500 steps × batch 32. On an L4 each step is short (frozen backbone + 360M decoder in bf16); plan ~5–15 minutes per run plus eval.

## 1. Install dependencies

In [1]:
%%capture
# Same pinning notes as in clip_pretrain.ipynb: do NOT upgrade torch/torchvision, do NOT pin numpy<2.
!pip -q install -U transformers datasets "sentence-transformers<4.0" accelerate tqdm matplotlib pyyaml einops
!pip -q install --force-reinstall --no-deps --no-cache-dir pillow==11.3.0

### 1b. (Optional) Install FlashAttention-2

FA2 is the writeup's recommended decoder attention implementation. SDPA works fine for this experiment (causal mask, no 4D mask needed) and avoids the ~5–10 min FA2 build. Skip this cell to use SDPA — the cell below sets `ATTN_IMPL` accordingly.

In [2]:
INSTALL_FLASH_ATTN = False  # flip to True if you want FA2

if INSTALL_FLASH_ATTN:
    !pip install -q flash-attn --no-build-isolation
    ATTN_IMPL = 'flash_attention_2'
else:
    ATTN_IMPL = 'sdpa'
print('decoder attn_implementation =', ATTN_IMPL)

decoder attn_implementation = sdpa


## 2. Mount Drive and set up the repo path

In [3]:
import os, sys
from pathlib import Path

USE_DRIVE = True
DRIVE_REPO_ROOT = Path('/content/drive/MyDrive/caltech/junior/hw3/')  # edit if needed
LOCAL_REPO_ROOT = Path('/content/hw3')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    REPO_ROOT = DRIVE_REPO_ROOT
else:
    REPO_ROOT = LOCAL_REPO_ROOT

assert REPO_ROOT.exists(), f'Repo root does not exist: {REPO_ROOT}'
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

Mounted at /content/drive
cwd = /content/drive/MyDrive/caltech/junior/hw3


In [4]:
import torch, torchvision

print('torch:', torch.__version__, '| torchvision:', torchvision.__version__, '| cuda:', torch.cuda.is_available())
torchvision.ops.nms(torch.zeros(0, 4), torch.zeros(0), 0.5)
print('torchvision ops OK')

torch: 2.10.0+cu128 | torchvision: 0.25.0+cu128 | cuda: True
torchvision ops OK


## 3. CLEVR-mini data

Runs `scripts/download_clevr.py` if `data/clevr_mini/` is missing. The dataset is small (10k examples).

In [5]:
CLEVR_ROOT = REPO_ROOT / 'data' / 'clevr_mini'
if not (CLEVR_ROOT / 'train.jsonl').exists():
    print(f'CLEVR not found at {CLEVR_ROOT} — downloading...')
    !python scripts/download_clevr.py
else:
    print(f'CLEVR ready at {CLEVR_ROOT}')

assert (CLEVR_ROOT / 'train.jsonl').exists(), 'download_clevr.py did not produce train.jsonl'
assert (CLEVR_ROOT / 'val.jsonl').exists()
assert (CLEVR_ROOT / 'images').exists()

CLEVR not found at /content/drive/MyDrive/caltech/junior/hw3/data/clevr_mini — downloading...
https://drive.google.com/file/d/1KsswLqfYLl1d91pg5kGUgwtPslo8njTB/view?usp=sharing
Downloaded 100/1703 MiB
Downloaded 200/1703 MiB
Downloaded 300/1703 MiB
Downloaded 400/1703 MiB
Downloaded 500/1703 MiB
Downloaded 600/1703 MiB
Downloaded 700/1703 MiB
Downloaded 800/1703 MiB
Downloaded 900/1703 MiB
Downloaded 1000/1703 MiB
Downloaded 1100/1703 MiB
Downloaded 1200/1703 MiB
Downloaded 1300/1703 MiB
Downloaded 1400/1703 MiB
Downloaded 1500/1703 MiB
Downloaded 1600/1703 MiB
Downloaded 1700/1703 MiB
Extracting to data
Done. CLEVR-mini available at data/clevr_mini


## 4. Configure paths and load the §5 config

In [6]:
import yaml
from pathlib import Path

CONFIG_PATH = REPO_ROOT / 'configs' / 'vlm_clevr.yaml'
PRETRAINED_VIT = REPO_ROOT / 'runs' / 'clip_eurosat' / 'best.pt'

assert CONFIG_PATH.exists(), f'config missing: {CONFIG_PATH}'
assert PRETRAINED_VIT.exists(), (
    f'CLIP-pretrained checkpoint missing at {PRETRAINED_VIT}. Run §3 first.'
)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Problem-specific overrides (writeup §5 injection-compare): 500 steps, bs=32, lr=1e-4.
NUM_STEPS = 500
BATCH_SIZE = 32
GRAD_ACCUM = 1
LR = 1.0e-4
FREEZE_CONFIG = 'A'   # projector only — ViT + decoder frozen
MASK_MODE = 'causal'

print(f'config         : {CONFIG_PATH}')
print(f'pretrained ViT : {PRETRAINED_VIT}')
print(f'decoder        : {cfg["decoder"]["model_name"]}  (attn={ATTN_IMPL})')
print(f'num_steps={NUM_STEPS}  batch_size={BATCH_SIZE}  lr={LR}  freeze={FREEZE_CONFIG}')

config         : /content/drive/MyDrive/caltech/junior/hw3/configs/vlm_clevr.yaml
pretrained ViT : /content/drive/MyDrive/caltech/junior/hw3/runs/clip_eurosat/best.pt
decoder        : HuggingFaceTB/SmolLM2-360M-Instruct  (attn=sdpa)
num_steps=500  batch_size=32  lr=0.0001  freeze=A


## 5. Train all three injection strategies

Each run writes `metrics.json` + `projector.pt` to `/content/runs/vlm_<injection>_<mask>_<freeze>/`, then single-shot copies them to Drive. Resumable: if `metrics.json` already exists on Drive for an injection mode and `OVERWRITE = False`, it's reused.

In [7]:
import argparse, json, shutil, time

from scripts.train_vlm import train, print_table, _default_run_dir

OVERWRITE = False
INJECTIONS = ['cls', 'all_patches', 'interleaved']

LOCAL_RUNS = Path('/content/runs')
LOCAL_RUNS.mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

all_metrics = {}
for injection in INJECTIONS:
    local_dir = _default_run_dir(LOCAL_RUNS, injection, MASK_MODE, FREEZE_CONFIG)
    drive_dir = REPO_ROOT / 'runs' / local_dir.name
    drive_metrics_path = drive_dir / 'metrics.json'

    if not OVERWRITE and drive_metrics_path.exists():
        all_metrics[injection] = json.loads(drive_metrics_path.read_text())
        print(f'[{injection}] reusing cached metrics  '
              f'(val_acc={all_metrics[injection]["final_val_acc"]:.4f})')
        continue

    local_dir.mkdir(parents=True, exist_ok=True)
    args = argparse.Namespace(
        config=CONFIG_PATH,
        pretrained_vit=PRETRAINED_VIT,
        clevr_root=CLEVR_ROOT,
        injection=injection,
        mask_mode=MASK_MODE,
        freeze_config=FREEZE_CONFIG,
        num_steps=NUM_STEPS,
        batch_size=BATCH_SIZE,
        grad_accum=GRAD_ACCUM,
        lr=LR,
        max_len=128,
        attn_impl=ATTN_IMPL,
        output_dir=local_dir,
        device=device,
        summarize=False,
        runs_root=LOCAL_RUNS,
    )
    print(f'\n========== injection={injection} ==========')
    t0 = time.time()
    m = train(args, cfg)
    print(f'[{injection}] wall = {time.time() - t0:.1f}s  val_acc={m["final_val_acc"]:.4f}  '
          f'peak={m["peak_mem_mb"]:.1f}MB  sec/step={m["sec_per_step"]:.3f}')
    all_metrics[injection] = m

    drive_dir.mkdir(parents=True, exist_ok=True)
    for f in local_dir.glob('*'):
        shutil.copy2(f, drive_dir / f.name)
    print(f'synced {local_dir} -> {drive_dir}')

print('\n========== summary ==========')
print_table(all_metrics)


========== injection=cls ==========


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...


step 200: val_overall=0.0540  loss=0.8238

step 400: val_overall=0.0980  loss=0.6756

step 500: val_overall=0.1420  loss=0.5348
saved /content/runs/vlm_all_patches_causal_A/metrics.json
[all_patches] wall = 165.1s  val_acc=0.1420  peak=6729.8MB  sec/step=0.286
synced /content/runs/vlm_all_patches_causal_A -> /content/drive/MyDrive/caltech/junior/hw3/runs/vlm_all_patches_causal_A

========== injection=interleaved ==========


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


[interleaved/causal/A] trainable=2,066,880  total=374,626,752


[interleaved]:   0%|          | 0/500 [00:00<?, ?it/s]


step 200: val_overall=0.0120  loss=1.1323

step 400: val_overall=0.0200  loss=0.8781

step 500: val_overall=0.0200  loss=0.6443
saved /content/runs/vlm_interleaved_causal_A/metrics.json
[interleaved] wall = 162.2s  val_acc=0.0200  peak=6730.4MB  sec/step=0.285
synced /content/runs/vlm_interleaved_causal_A -> /content/drive/MyDrive/caltech/junior/hw3/runs/vlm_interleaved_causal_A

========== summary ==========

     injection |  n_visual |   val_acc |  peak_mem_MB | sec_per_step
--------------------------------------------------------------------
           cls |         1 |    0.3460 |       3429.4 |        0.249
   all_patches |        65 |    0.1420 |       6729.8 |        0.286
   interleaved |        65 |    0.0200 |       6730.4 |        0.285



## 6. Re-print the deliverable table

Reads each injection's `metrics.json` from Drive and prints the same 3-row table. Survives kernel restarts.

In [8]:
import json
from scripts.train_vlm import print_table, _default_run_dir

saved = {}
for inj in ('cls', 'all_patches', 'interleaved'):
    p = _default_run_dir(REPO_ROOT / 'runs', inj, MASK_MODE, FREEZE_CONFIG) / 'metrics.json'
    if p.exists():
        saved[inj] = json.loads(p.read_text())
    else:
        print(f'[warn] missing {p}')
print_table(saved)


     injection |  n_visual |   val_acc |  peak_mem_MB | sec_per_step
--------------------------------------------------------------------
           cls |         1 |    0.3460 |       3429.4 |        0.249
   all_patches |        65 |    0.1420 |       6729.8 |        0.286
   interleaved |        65 |    0.0200 |       6730.4 |        0.285



## 7. Writeup notes (1 paragraph)

Fill in using your actual numbers. Things to address:

- **Which injection wins on val accuracy.** `all_patches` (or `interleaved`, which injects the same patch sequence at a different position) typically beats `cls` by a meaningful margin on CLEVR. CLEVR questions like "how many red metal spheres?" require spatial / per-object information that the single CLS vector compresses away.

- **Cost columns.** `n_visual = 1` for `cls`, `n_visual = 65` for `all_patches` and `interleaved` (`(64/8)² + 1`). Peak memory and sec/step scale roughly with the squared sequence length of the decoder, since attention is O(T²) — expect `all_patches`/`interleaved` to use noticeably more memory and time than `cls`.

- **Is the extra cost worth it?** Compare the accuracy delta against the memory/time delta. Typically yes for CLEVR: the win from giving the decoder access to per-patch features dwarfs the extra compute, and patch-level features are what's missing from a pooled CLS vector.

- **Connection to the ViT-pooling problem.** This is the same trade-off restated: CLS pooling loses spatial information, average / patch-level pooling preserves it. For a downstream task that *needs* spatial structure (counting, attribute binding), patch-level injection is the VLM analogue of using all tokens instead of just CLS.

Optional: comment on whether `interleaved` and `all_patches` produced essentially the same accuracy. They should — the visual content is identical; only the position in the text stream differs.